# Pharmacon Anonymizer: Automated Prescription PII/PHI Redaction Pipeline
**Goal:** Layout-agnostic de-identification of medical prescriptions (Digital Vector PDFs, Scanned Slips, EMR Exports).
- Strips all 18 HIPAA PII/PHI identifiers (Patient Name, Doctor Name, Contact, Address, UHID/MRN, Dates).
- Strictly preserves all clinical pharmacology data (Drug name, dosage, frequency, directions).
- Dual-Engine Architecture: Fast PyMuPDF Vector Redactor + Florence-2 Visual Grounding on T4 GPU.

In [ ]:
# Cell 1: Environment Setup on Google Colab (Enable GPU in Runtime -> Change runtime type -> T4)
!pip install -q pymupdf transformers accelerate timm pillow opencv-python-headless
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: Download Sample Prescriptions (Real Scanned + Digital Vector)
!mkdir -p samples
!wget -q https://raw.githubusercontent.com/abhijeetk597/medical-data-extraction/main/backend/resources/prescription/pre_1.pdf -O samples/scanned_rx_1.pdf
!wget -q https://raw.githubusercontent.com/abhijeetk597/medical-data-extraction/main/backend/resources/prescription/pre_2.pdf -O samples/scanned_rx_2.pdf
!wget -q https://raw.githubusercontent.com/abhijeetk597/medical-data-extraction/main/backend/resources/patient_details/pd_1.pdf -O samples/patient_record_1.pdf
print("Downloaded sample test files into samples/")
!ls -lh samples/


In [ ]:
# Cell 3: Load Florence-2 Vision-Language Grounding Model
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import re

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "microsoft/Florence-2-base"
print(f"Loading {model_id} on {device}...")
florence_model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float16 if device=="cuda" else torch.float32).to(device).eval()
florence_processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
print("Florence-2 loaded successfully!")


In [ ]:
# Cell 4: Visual Grounding Redactor Function
import cv2
import numpy as np

def ground_and_redact_image(image_path, output_path, padding=8):
    image = Image.open(image_path).convert("RGB")
    w_orig, h_orig = image.size
    
    # Prompt Florence-2 to ground PII elements
    prompt = "<CAPTION_TO_PHRASE_GROUNDING> patient name . doctor name . address . phone number . date of visit . signature . clinic header ."
    inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(device)
    
    if device == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
        
    with torch.no_grad():
        generated_ids = florence_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=512,
            num_beams=3
        )
    
    generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = florence_processor.post_process_generation(
        generated_text,
        task="<CAPTION_TO_PHRASE_GROUNDING>",
        image_size=(w_orig, h_orig)
    )
    
    img_cv = cv2.imread(image_path)
    bboxes = parsed_answer.get("<CAPTION_TO_PHRASE_GROUNDING>", {}).get("bboxes", [])
    labels = parsed_answer.get("<CAPTION_TO_PHRASE_GROUNDING>", {}).get("labels", [])
    
    redacted_count = 0
    for bbox, label in zip(bboxes, labels):
        x0, y0, x1, y1 = [int(v) for v in bbox]
        # Apply safety padding
        x0 = max(0, x0 - padding)
        y0 = max(0, y0 - padding)
        x1 = min(w_orig, x1 + padding)
        y1 = min(h_orig, y1 + padding)
        
        # Draw solid black mask
        cv2.rectangle(img_cv, (x0, y0), (x1, y1), (0, 0, 0), -1)
        redacted_count += 1
        
    cv2.imwrite(output_path, img_cv)
    return {"redacted_count": redacted_count, "labels": labels, "bboxes": bboxes}
print("ground_and_redact_image function ready!")


In [ ]:
# Cell 5: Run Anonymization on Test Images & Display Side-by-Side
import matplotlib.pyplot as plt
import pymupdf

# Convert PDF page to image
doc = pymupdf.open("samples/scanned_rx_1.pdf")
pix = doc[0].get_pixmap(dpi=150)
pix.save("samples/scanned_rx_1_input.png")
doc.close()

res = ground_and_redact_image("samples/scanned_rx_1_input.png", "samples/scanned_rx_1_redacted.png")
print("Detected & Redacted Entities:", res["labels"])

fig, axes = plt.subplots(1, 2, figsize=(14, 9))
axes[0].imshow(Image.open("samples/scanned_rx_1_input.png"))
axes[0].set_title("Original Prescription (With PII)")
axes[0].axis("off")

axes[1].imshow(Image.open("samples/scanned_rx_1_redacted.png"))
axes[1].set_title("Anonymized Prescription (PII Black-Boxed)")
axes[1].axis("off")
plt.tight_layout()
plt.show()
